<a href="https://colab.research.google.com/github/HumphreyChimanya/Solwezi_city_council_Data/blob/main/Copy_of_Solwezi_City_Council_DataAnalytics_Assignment_IntegrationAndTransformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Integration and Transformation

### Subtask:
Determine if any of the loaded DataFrames need to be merged, joined, or transformed (e.g., pivot tables, aggregations) to create a comprehensive dataset for analysis.



To prepare for data integration, we need to identify common columns across the dataframes. Printing the columns of each dataframe will help in this identification process.



In [ ]:
print("--- Columns in each DataFrame ---")
for df_name, df in dfs.items():
    print(f"\nDataFrame '{df_name}':")
    print(df.columns.tolist())

--- Columns in each DataFrame ---

DataFrame 'table_3':
['No.', 'Project Description', 'Sector', 'Ward', 'Zone', "Engineer's estimates", 'Allocated amount by CDFC', 'Comment']

DataFrame 'table_1':
['No.', 'Name of Group', 'Type of Group (Youth/ Women/ Community)', 'Ward', 'Zone', 'Type of Venture(specify)', 'Sector', 'Amount Requested', 'Amount Recommended by CDF']

DataFrame 'table_4':
['No.', 'Project Description', 'Sector', 'Ward', 'Zone', "Engineer's estimates", 'Allocated amount by CDFC', 'Comment']

DataFrame 'table_2':
['S/NO', 'NAME OF CLUB OR COOPERATIVE OR COMPANY', 'DESCRIPTION OF PROJECT', 'CONTACT PERSON (NAME)', 'WARD', 'AMOUNT REQUESTED', 'AMOUNT RECOMMENDED BY CDF']



Standardizing column names for data integration. We will define a mapping of existing column names to more consistent, snake_case names and apply this mapping to all DataFrames in the `dfs` dictionary. This ensures that columns with similar content can be easily identified and merged.



In [ ]:
column_mapping = {
    'No.': 'project_no',
    'S/NO': 'project_no',
    'Project Description': 'project_description',
    'DESCRIPTION OF PROJECT': 'project_description',
    'Sector': 'sector',
    'Ward': 'ward',
    'WARD': 'ward',
    'Zone': 'zone',
    "Engineer's estimates": 'engineers_estimates',
    'Allocated amount by CDFC': 'allocated_amount_cdf',
    'Comment': 'comment',
    'Name of Group': 'group_name',
    'NAME OF CLUB OR COOPERATIVE OR COMPANY': 'group_name',
    'Type of Group (Youth/ Women/ Community)': 'group_type',
    'Type of Venture(specify)': 'venture_type',
    'Amount Requested': 'amount_requested',
    'AMOUNT REQUESTED': 'amount_requested',
    'Amount Recommended by CDF': 'amount_recommended_cdf',
    'AMOUNT RECOMMENDED BY CDF': 'amount_recommended_cdf',
    'CONTACT PERSON (NAME)': 'contact_person'
}

for df_name, df in dfs.items():
    # Convert all column names to uppercase for case-insensitive mapping
    df.columns = df.columns.str.title().str.replace('_', ' ').str.replace('.', '').str.strip() # Normalizing to make matching easier

    # Apply mapping only if the column exists
    new_columns = []
    for col in df.columns:
        mapped_col = None
        for old_col_key, new_col_value in column_mapping.items():
            if old_col_key.replace('.', '').replace('_', ' ').strip().title() == col:
                mapped_col = new_col_value
                break
        if mapped_col:
            new_columns.append(mapped_col)
        else:
            new_columns.append(col.lower().replace(' ', '_')) # Default to lowercase snake_case

    dfs[df_name].columns = new_columns
    print(f"\n--- Standardized columns for DataFrame: '{df_name}' ---")
    print(dfs[df_name].columns.tolist())



--- Standardized columns for DataFrame: 'table_3' ---
['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment']

--- Standardized columns for DataFrame: 'table_1' ---
['project_no', 'group_name', 'group_type', 'ward', 'zone', 'venture_type', 'sector', 'amount_requested', 'amount_recommended_cdf']

--- Standardized columns for DataFrame: 'table_4' ---
['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment']

--- Standardized columns for DataFrame: 'table_2' ---
['project_no', 'group_name', 'project_description', 'contact_person', 'ward', 'amount_requested', 'amount_recommended_cdf']



We need to identify common and unique columns across the dataframes to determine suitable merging strategies. This involves listing all column names and then finding their intersections and differences.



In [ ]:
all_columns = set()
for df_name, df in dfs.items():
    all_columns.update(df.columns)

print("--- Common and Unique Columns Across DataFrames ---")
for df_name, df in dfs.items():
    print(f"\nDataFrame '{df_name}':")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Common with others: {list(set(df.columns) & all_columns - set(df.columns))}")
    print(f"  Unique to this DataFrame: {list(set(df.columns) - (all_columns - set(df.columns)))}")

# Identify truly common columns for merging
common_cols = set.intersection(*(set(df.columns) for df in dfs.values()))
print(f"\nTruly common columns across ALL DataFrames: {list(common_cols)}")

--- Common and Unique Columns Across DataFrames ---

DataFrame 'table_3':
  Columns: ['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment']
  Common with others: []
  Unique to this DataFrame: ['sector', 'zone', 'project_no', 'allocated_amount_cdf', 'ward', 'comment', 'project_description', 'engineers_estimates']

DataFrame 'table_1':
  Columns: ['project_no', 'group_name', 'group_type', 'ward', 'zone', 'venture_type', 'sector', 'amount_requested', 'amount_recommended_cdf']
  Common with others: []
  Unique to this DataFrame: ['sector', 'venture_type', 'zone', 'amount_requested', 'project_no', 'group_type', 'amount_recommended_cdf', 'ward', 'group_name']

DataFrame 'table_4':
  Columns: ['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment']
  Common with others: []
  Unique to this DataFrame: ['sector', 'zone', 'project_no', 'allocated_amount_cdf', 'ward

We will merge `table_3` and `table_4` based on their common columns to create an integrated `projects_df`.



In [ ]:
print("--- Merging table_3 and table_4 into projects_df ---")
projects_df = pd.merge(dfs['table_3'], dfs['table_4'], on=list(dfs['table_3'].columns), how='outer', suffixes=('_3', '_4'))

print("Projects DataFrame created with merged data:")
display(projects_df.head())

print("\nProjects DataFrame columns:")
print(projects_df.columns.tolist())

print("\nProjects DataFrame info:")
projects_df.info()

--- Merging table_3 and table_4 into projects_df ---
Projects DataFrame created with merged data:


,project_no,project_description,sector,ward,zone,engineers_estimates,allocated_amount_cdf,comment
0,1,Fuel For Feeder Roads,Transport,All,All,158368.0,3278013.69,Presidential Directive
1,2,Purchase Of Medical And Non-Medical Equipments,Health,Sandangombe,"Kandakanda,",158368.0,158368.00,Recommended By Cdfc
2,3,Purchase Of A Caterpillar 140-01A Gc Motor Grader,Unknown,All,All,158368.0,158368.00,Recommended By Cdfc
3,4,Construction Of A Double Storey Crb At Solwezi...,Education,Kimasala,Kimasala,2500000.0,2100000.00,Recommended By Cdfc
4,5,Construction Of A Double Storey Crb At Rodwel ...,Education,Kyawama,Kyawama,2500000.0,2100000.00,Recommended By Cdfc



Projects DataFrame columns:
['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment']

Projects DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   project_no            9 non-null      Int64  
 1   project_description   15 non-null     object 
 2   sector                15 non-null     object 
 3   ward                  15 non-null     object 
 4   zone                  15 non-null     object 
 5   engineers_estimates   15 non-null     float64
 6   allocated_amount_cdf  15 non-null     float64
 7   comment               15 non-null     object 
dtypes: Int64(1), float64(2), object(5)
memory usage: 1.1+ KB


We will merge `table_1` and `table_2` based on their common columns (`project_no`, `group_name`, `ward`, `amount_requested`, `amount_recommended_cdf`) to create a combined DataFrame for group-related projects.



In [ ]:
print("--- Merging table_1 and table_2 into groups_df ---")

common_cols_t1_t2 = list(set(dfs['table_1'].columns) & set(dfs['table_2'].columns))
groups_df = pd.merge(dfs['table_1'], dfs['table_2'], on=common_cols_t1_t2, how='outer', suffixes=('_t1', '_t2'))

print("Groups DataFrame created with merged data:")
display(groups_df.head())

print("\nGroups DataFrame columns:")
print(groups_df.columns.tolist())

print("\nGroups DataFrame info:")
groups_df.info()

--- Merging table_1 and table_2 into groups_df ---
Groups DataFrame created with merged data:


,project_no,group_name,group_type,ward,zone,venture_type,sector,amount_requested,amount_recommended_cdf,project_description,contact_person
0,26,Jikwachi Multi- Purpose Cooperative,Community,Kazhiba,Kazhiba,Poultry,Agriculture,3650.0,15000.0,NaN,NaN
1,152,Kyampeku Evangelical Women Club,Women,Sandangombe,Sandangombe,Poultry,Agriculture,10000.0,10000.0,NaN,NaN
2,155,Nsulo Ya Katuba New Covenant Club,Youth,Sandangombe,Sandagombe,Poultry,Agriculture,10000.0,10000.0,NaN,NaN
3,<NA>,Twisekelemo Mpcs,Unknown,Unknown,Unknown,Unknown,Unknown,11000.0,11000.0,NaN,NaN
4,117,Tebeth General Dealers,Company,Kyalalankuba,Kimakolwe,Poultry,Agriculture,15000.0,12000.0,NaN,NaN



Groups DataFrame columns:
['project_no', 'group_name', 'group_type', 'ward', 'zone', 'venture_type', 'sector', 'amount_requested', 'amount_recommended_cdf', 'project_description', 'contact_person']

Groups DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   project_no              156 non-null    Int64  
 1   group_name              233 non-null    object 
 2   group_type              140 non-null    object 
 3   ward                    233 non-null    object 
 4   zone                    140 non-null    object 
 5   venture_type            140 non-null    object 
 6   sector                  140 non-null    object 
 7   amount_requested        233 non-null    float64
 8   amount_recommended_cdf  233 non-null    float64
 9   project_description     93 non-null     object 
 10  contact_person          93 non-

We will merge `projects_df` and `groups_df` based on the common columns `project_no` and `ward` to create a final integrated DataFrame, `integrated_df`.



In [ ]:
print("--- Merging projects_df and groups_df into integrated_df ---")

# Identify common columns between projects_df and groups_df for merging
common_cols_projects_groups = list(set(projects_df.columns) & set(groups_df.columns))

integrated_df = pd.merge(projects_df, groups_df, on=common_cols_projects_groups, how='outer', suffixes=('_projects', '_groups'))

print("Integrated DataFrame created with merged data:")
display(integrated_df.head())

print("\nIntegrated DataFrame columns:")
print(integrated_df.columns.tolist())

print("\nIntegrated DataFrame info:")
integrated_df.info()

--- Merging projects_df and groups_df into integrated_df ---
Integrated DataFrame created with merged data:


,project_no,project_description,sector,ward,zone,engineers_estimates,allocated_amount_cdf,comment,group_name,group_type,venture_type,amount_requested,amount_recommended_cdf,contact_person
0,114,NaN,Agric-,Kyafukuma,Kamatete,NaN,NaN,NaN,Lwamitobo Mpc,Community,Gardening,40000.0,10000.0,NaN
1,39,NaN,Agriculture,Kimasala,Community,NaN,NaN,NaN,Mwazowetu Mpc,Community,Poultry,50000.0,25000.0,NaN
2,45,NaN,Agriculture,Kimasala,Community,NaN,NaN,NaN,Katubila Fwani Mpc,Women,Poultry,40000.0,20000.0,NaN
3,<NA>,NaN,Agriculture,Kifubwa,Kainafumu,NaN,NaN,NaN,Wonderful Women Cooperative,Women,Poultry,20000.0,10000.0,NaN
4,<NA>,NaN,Agriculture,Kifubwa,Kakombe,NaN,NaN,NaN,Chibwika Times Women Multi- Purpose,Women,Poultry,15000.0,10000.0,NaN



Integrated DataFrame columns:
['project_no', 'project_description', 'sector', 'ward', 'zone', 'engineers_estimates', 'allocated_amount_cdf', 'comment', 'group_name', 'group_type', 'venture_type', 'amount_requested', 'amount_recommended_cdf', 'contact_person']

Integrated DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   project_no              165 non-null    Int64  
 1   project_description     108 non-null    object 
 2   sector                  155 non-null    object 
 3   ward                    248 non-null    object 
 4   zone                    155 non-null    object 
 5   engineers_estimates     15 non-null     float64
 6   allocated_amount_cdf    15 non-null     float64
 7   comment                 15 non-null     object 
 8   group_name              233 non-null    object 
 9   group_type        